# **Author:** Mukhamedali Daniyaruly
# **Topic:** Training Vision Language Models: SigLIP + C Abstractor + KazLM LoRA
# **Date:** 23/01/2026

In [ ]:
!pip install -U bitsandbytes transformers peft accelerate

In [ ]:
import os
import math
import glob
import random
import shutil
from PIL import Image
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
import wandb
from google.colab import drive

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, random_split

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoProcessor, AutoModel, BitsAndBytesConfig
from peft import get_peft_model, PeftConfig, LoraConfig, prepare_model_for_kbit_training

# PEFT

In [ ]:
# We need to verify us to get access for using KazLM model
from huggingface_hub import login
login("YOUR_HUGGINGFACE_TOKEN")

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("issai/LLama-3.1-KazLLM-1.0-8B")
model = AutoModelForCausalLM.from_pretrained("issai/LLama-3.1-KazLLM-1.0-8B", quantization_config=bnb_config)

In [ ]:
peft_model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    lora_alpha=16
)

peft_model = get_peft_model(peft_model, lora_config)

In [ ]:
peft_model.print_trainable_parameters()

# Configuration

In [ ]:
drive.mount("/content/drive")
class Configuration:
  def __init__(self):
    # Paths
    self.data_path = "/content/drive/MyDrive/Vision KazLM/COCO_Data/coco_kazakh_train_FULL.json"
    self.images_path = "/content/drive/MyDrive/Vision KazLM/COCO_Data/images"
    self.save_path = "/content/drive/MyDrive/Vision KazLM/3types/lora/training_checkpoint"

    # Models
    self.llm_model_name = "issai/LLama-3.1-KazLLM-1.0-8B"
    self.vision_model_name = "google/siglip-so400m-patch14-384"

    # Hyperparameters
    self.vision_embed_dim = 1152
    self.llm_embed_dim = 4096
    self.max_length = 128
    self.epochs = 1
    self.lr = 2e-5
    self.max_length = 128
    self.batch_size = 2
    self.grad_accumulation = 32
    self.log_steps = 2
    self.warmup_ratio = 0.05
    self.weight_decay = 0.01

    # device
    self.device = "cuda" if torch.cuda.is_available() else "cpu"

device = "cuda" if torch.cuda.is_available() else "cpu"
config = Configuration()
os.makedirs(config.save_path, exist_ok=True)

# Download Images

In [ ]:
import os

# Create a structure
root_dir = "/content/coco_karpathy"
os.makedirs(f"{root_dir}/images", exist_ok=True)
os.makedirs(f"{root_dir}/annotations", exist_ok=True)

# 2. Download images 2014 (Train + Val) ~13GB + 6GB
# We split train and validation images
!wget -c http://images.cocodataset.org/zips/train2014.zip -O {root_dir}/train2014.zip
!unzip -q {root_dir}/train2014.zip -d {root_dir}/images/
!rm {root_dir}/train2014.zip

!wget -c http://images.cocodataset.org/zips/val2014.zip -O {root_dir}/val2014.zip
!unzip -q {root_dir}/val2014.zip -d {root_dir}/images/
!rm {root_dir}/val2014.zip

# Dataset Class

In [ ]:
class VLMDataset(Dataset):
    def __init__(self, json_path, images_root, tokenizer, processor, max_length=128):
        """
        Args:
            json_path: Путь к JSON файлу с данными (coco_kazakh_train_FULL.json)
            images_root: Папка с картинками
            tokenizer: Токенизатор от LLM (KazLM)
            processor: Процессор от SigLIP
            max_length: Максимальная длина текста
        """
        self.images_root = images_root
        self.tokenizer = tokenizer
        self.processor = processor
        self.max_length = max_length
        self.prompt_text = "Суреттегі көрініс: " # Хардкодим промпт здесь

        print(f"Loading data from {json_path}...")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        # Фильтрация данных: оставляем только те, где есть подписи
        self.data = [entry for entry in raw_data if entry.get('captions_kk')]

        print(f"Original size: {len(raw_data)}")
        print(f"Filtered size (valid captions): {len(self.data)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        entry = self.data[idx]

        # --- 1. Обработка изображения ---
        image_path = os.path.join(self.images_root, entry['image_path'])

        try:
            image = Image.open(image_path).convert('RGB')
            # Обработка через SigLIP Processor
            # pixel_values shape: [3, 384, 384]
            pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze(0)
        except Exception as e:
            # Если картинка битая, берем случайную другую из датасета
            # print(f"Warning: Error loading image {image_path}, skipping...")
            new_idx = random.randint(0, len(self.data) - 1)
            return self.__getitem__(new_idx)

        # --- 2. Обработка текста ---
        # Выбираем одну случайную подпись из списка
        caption = random.choice(entry['captions_kk'])

        # Формируем полный текст: Промпт + Описание + EOS токен
        full_text = self.prompt_text + caption + self.tokenizer.eos_token

        # Токенизируем полный текст
        tokenized = self.tokenizer(
            full_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        input_ids = tokenized.input_ids.squeeze(0)
        attention_mask = tokenized.attention_mask.squeeze(0)

        # --- 3. Создание Labels (КРИТИЧЕСКИ ВАЖНО) ---
        labels = input_ids.clone()

        # Нам нужно замаскировать (сделать -100) всё, что относится к промпту.
        # Модель не должна учиться предсказывать фразу "Суреттегі көрініс: "

        # Токенизируем промпт отдельно, чтобы узнать его точную длину в токенах.
        # add_special_tokens=True, так как в начале full_text тоже стоит BOS токен.
        prompt_ids = self.tokenizer(self.prompt_text, add_special_tokens=True, truncation=True, max_length=self.max_length).input_ids
        prompt_len = len(prompt_ids)

        # 1. Маскируем промпт
        # Если промпт длиннее max_length (вряд ли), обрезаем по границе
        safe_prompt_len = min(prompt_len, self.max_length)
        labels[:safe_prompt_len] = -100

        # 2. Маскируем паддинги (пустые места в конце)
        labels[input_ids == self.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values,   # Для Vision Encoder
            "input_ids": input_ids,         # Для LLM Input
            "attention_mask": attention_mask, # Маска внимания
            "labels": labels                # Для расчета Loss
        }

In [ ]:
def collate_fn(batch):
    # Batch это список словарей: [dict1, dict2, ...]
    # Нам нужно сделать словарь списков тензоров: {key: stack([t1, t2])}

    pixel_values = torch.stack([item['pixel_values'] for item in batch])
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    labels = torch.stack([item['labels'] for item in batch])

    return {
        "pixel_values": pixel_values,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
images_path = os.path.join("coco_karpathy", "images")

In [ ]:
tokenizer.pad_token_id = tokenizer.eos_token_id
full_dataset = VLMDataset(
    json_path=config.data_path, # Path to json file
    images_root=images_path, # Images path
    tokenizer=tokenizer, # Tokenizer
    processor=processor,   # Processor

)

In [ ]:
train_size = int(0.9 * len(full_dataset))
val_size = int(0.05 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
for batch in train_loader:
  px, ins, masks, lbs = batch["pixel_values"], batch["input_ids"], batch["attention_mask"], batch["labels"]

  print(px.shape)
  print(ins.shape)
  print(masks.shape)
  print(lbs.shape)

  break

In [ ]:
# Запусти это в отдельной ячейке ПРЯМО ВО ВРЕМЯ ОБУЧЕНИЯ
# Это прогонит всего 20 батчей валидации и скажет правду.

print("Running Emergency Validation Check...")
model.eval()
temp_val_loss = 0
steps_to_check = 20  # Проверим только 20 батчей, чтобы быстро

with torch.no_grad():
    for i, batch in enumerate(val_loader):
        if i >= steps_to_check: break

        pixel_values = batch["pixel_values"].to(device, dtype=torch.bfloat16)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(pixel_values, input_ids, attention_mask, labels)

        temp_val_loss += outputs.loss.item()

print(f"Sanity Check Loss (over {steps_to_check} batches): {temp_val_loss / steps_to_check:.4f}")

# Возвращаем модель в режим тренировки
model.train()

# Model Architecture

In [ ]:
class VisionEncoder(nn.Module):
  def __init__(self, vision_encoder):
    super.__init__()

    self.vision_encoder = vision_encoder

    for p in self.vision_encoder.parameters():
      p.requires_grad=False

  def forward(self, pixel_values):
    outs = self.vision_encoder(pixel_values)
    return outs["last_hidden_state"]

In [ ]:
class CAbstractor(nn.Module):
  def __init__(self, vision_embeds, llm_embeds):
    super().__init__()

    # 1. Входная размерность увеличилась в 4 раза из-за сжатия 2x2
    self.input_dim = vision_embeds * 4  # 1152 * 4 = 4608
    self.output_dim = llm_embeds        # 4096

    # 2. Простой и надежный MLP вместо сложного SwiGLU
    self.model = nn.Sequential(
        nn.Linear(self.input_dim, self.output_dim),
        nn.GELU(),
        nn.Linear(self.output_dim, self.output_dim)
    )

  def forward(self, images):
    # images shape: [Batch, 729, 1152]
    B, L, D = images.size()

    # Считаем размеры для сжатия
    h = w = int(L ** 0.5) # 27
    new_h = new_w = (h - 1) // 2 # 13

    # Твой алгоритм сжатия (он правильный, мы просто отрезаем 27-й пиксель)
    images = images.view(B, h, w, D)
    images = images[:, :new_h*2, :new_w*2, :] # Crop to 26x26 cleanly
    images = images.reshape(B, new_h, 2, new_w, 2, D)
    images = images.permute(0, 1, 3, 2, 4, 5) # [B, 13, 13, 2, 2, D]

    # Flattening 2x2 blocks: [B, 169, 4608]
    images = images.reshape(B, new_h * new_w, 4 * D)

    # Прогоняем через MLP
    return self.model(images)

In [ ]:
class KazVLM(nn.Module):
  def __init__(self, vision_model, llm_model, config):
      super().__init__()
      self.vision_encoder = vision_model
      self.llm = llm_model

      self.abstractor = CAbstractor(config.vision_embed_dim, config.llm_embed_dim, mlp_ratio=1)

      # Freeze Vision & LLM (Base weights)
      for param in self.vision_encoder.parameters():
          param.requires_grad = False

      self.llm.gradient_checkpointing_enable()

      print(f"Model Initialized with C-Abstractor.")
      print(f"Trainable Params in Abstractor: {sum(p.numel() for p in self.abstractor.parameters())}")

  def forward(self, pixel_values, input_ids, attention_mask, labels=None):
      # 1. Get Image Embeddings [Batch, 729, 1152]
      with torch.no_grad():
          vision_outputs = self.vision_encoder(pixel_values)
          image_features = vision_outputs.last_hidden_state

      # 2. Apply C-Abstractor: [Batch, 729, 1152] -> [Batch, 169, 4096]
      # ВОТ ТУТ МАГИЯ
      image_embeds = self.abstractor(image_features)

      # 3. Get Text Embeddings
      text_embeds = self.llm.get_input_embeddings()(input_ids)

      # 4. Concatenate
      inputs_embeds = torch.cat([image_embeds, text_embeds], dim=1)

      # 5. Adjust Attention Mask (Теперь у нас 169 токенов картинки, а не 729!)
      batch_size = input_ids.shape[0]
      image_mask = torch.ones((batch_size, image_embeds.shape[1]), device=image_embeds.device) # 169 ones
      full_attention_mask = torch.cat([image_mask, attention_mask], dim=1)

      # 6. Adjust Labels
      if labels is not None:
          image_labels = torch.full((batch_size, image_embeds.shape[1]), -100, device=labels.device)
          full_labels = torch.cat([image_labels, labels], dim=1)
      else:
          full_labels = None

      # 7. Pass to LLM
      outputs = self.llm(
          inputs_embeds=inputs_embeds,
          attention_mask=full_attention_mask,
          labels=full_labels
      )

      return outputs

# Download Models: SigLIP + KazLM

In [ ]:
# We need to verify us to get access for using KazLM model
from huggingface_hub import login
login("YOUR_HUGGINGFACE_TOKEN")

In [ ]:
# Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

In [ ]:
# Download tokenizer and set padding token
tokenizer = AutoTokenizer.from_pretrained(config.llm_model_name)
tokenizer_pad_token = tokenizer.eos_token

# Download processor
processor = AutoProcessor.from_pretrained(config.vision_model_name)

In [ ]:
# Download SiLIP model and set to cuda
vision_model = AutoModel.from_pretrained(
    config.vision_model_name,
    torch_dtype=torch.float16,
    devic_map="cuda"
)

In [ ]:
# Download KazLM and Quantize it
llm_model = AutoModelForCausalLM.from_pretrained(config.llm_model_name, quantization_config=bnb_config, device_map="cuda")

# Training configuration

In [ ]:
# Initialize a Vision Language Model
model = KazVLM(vision_model, llm_model)
model = model.to(device)

In [ ]:
# optimizer
num_training_steps = int((1 * config.epochs) // config.grad_accumulation)
num_warmup_steps = int(num_training_steps * config.warmup_ratio)

scaler = torch.cuda.amp.GradScaler()
params_to_optimize = list(model.abstractor.parameters()) + list(model.llm.parameters())
optimizer = torch.optim.AdamW(params_to_optimize, lr=2e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs * len(train_loader))

In [ ]:
wandb.init(
    project="uncategorized",
    name="SigLIP_Abstractor_LoRA_r32",

    config={
        "learning_rate": config.lr,
        "batch_size": config.batch_size * config.grad_accumulation,
        "epochs": config.epochs,
        "architecture": "SigLIP + C-Abstractor + LoRA KazLM",
        "dataset": "COCO-Kaz-113k",
        "lora_r": 32,
        "abstractor_type": "MLP"
    }
)

In [ ]:
# Настройка частоты
LOG_STEPS = 10
EVAL_STEPS = 125
SAVE_STEPS = 125

# Инициализация
model.train()
optimizer.zero_grad()
scaler = torch.cuda.amp.GradScaler()

# Для статистики
total_loss = 0.0
current_loss = 0.0
global_step = 0
best_val_loss = float('inf')

print(f"START TRAINING on {device} | Batch: {config.batch_size} | Accumulation: {config.grad_accumulation}")

for epoch in range(config.epochs):

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.epochs}")

    for step, batch in enumerate(pbar):
        # 1. Move to Device
        pixel_values = batch["pixel_values"].to(device, dtype=torch.bfloat16)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # 2. Forward Pass
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(pixel_values, input_ids, attention_mask, labels)
            loss = outputs.loss
            loss_scaled = loss / config.grad_accumulation

        # 3. Backward
        scaler.scale(loss_scaled).backward()
        current_loss += loss.item()

        # 4. Optimizer Step
        if (step + 1) % config.grad_accumulation == 0:
            scaler.unscale_(optimizer)

            # !!! FIX 1: Используем model.abstractor, а не projector !!!
            # Также нам нужно клипать градиенты и у LoRA тоже, если мы её тюним?
            # Но обычно клипают всё вместе. Пока оставим только abstractor,
            # так как LoRA сама по себе стабильна.
            torch.nn.utils.clip_grad_norm_(model.abstractor.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

            global_step += 1

            # --- LOGGING ---
            if global_step % LOG_STEPS == 0:
                avg_loss = current_loss / (LOG_STEPS * config.grad_accumulation)
                try:
                    ppl = math.exp(avg_loss)
                except OverflowError:
                    ppl = float('inf')

                # Update WandB
                wandb.log({
                    "train_loss": avg_loss,
                    "train_ppl": ppl,
                    "lr": scheduler.get_last_lr()[0],
                    "epoch": epoch + (step / len(train_loader)),
                    "global_step": global_step  # Явно указываем шаг для оси X
                })

                pbar.set_description(f"Ep {epoch+1} | Loss: {avg_loss:.4f} | PPL: {ppl:.2f}")
                current_loss = 0.0

            # --- SAVING & VALIDATION ---
            if global_step % EVAL_STEPS == 0:
                print(f"\n🔍 Running Validation at step {global_step}...")
                model.eval()
                val_loss = 0
                val_steps = 0

                with torch.no_grad():
                    for val_batch in tqdm(val_loader, leave=False):
                        v_px = val_batch["pixel_values"].to(device, dtype=torch.bfloat16)
                        v_in = val_batch["input_ids"].to(device)
                        v_att = val_batch["attention_mask"].to(device)
                        v_lbl = val_batch["labels"].to(device)

                        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                            v_out = model(v_px, v_in, v_att, v_lbl)
                            val_loss += v_out.loss.item()
                            val_steps += 1

                avg_val = val_loss / val_steps
                val_ppl = math.exp(avg_val) if avg_val < 20 else float('inf')

                print(f"✅ Val Loss: {avg_val:.4f} | PPL: {val_ppl:.2f}")

                # !!! FIX 2: Добавляем global_step, чтобы точки на графике совпадали с train_loss
                wandb.log({
                    "val_loss": avg_val,
                    "val_ppl": val_ppl,
                    "global_step": global_step
                })

                # Save Best
                if avg_val < best_val_loss:
                    best_val_loss = avg_val
                    # !!! FIX 3: model.abstractor !!!
                    torch.save(model.abstractor.state_dict(), f"{config.save_path}/best_abstractor.pth")
                    model.llm.save_pretrained(f"{config.save_path}/best_lora_adapter")
                    print("💾 Best Model Saved (Abstractor + LoRA)!")

                # Save Regular Checkpoint
                # !!! FIX 4: Используем global_step в имени файла !!!
                torch.save({
                    "step": global_step,
                    "c_abstractor": model.abstractor.state_dict(),
                    "optimizer": optimizer.state_dict()
                }, f"{config.save_path}/{global_step}_checkpoint.pth")

                model.llm.save_pretrained(f"{config.save_path}/{global_step}_lora_adapter")

                model.train()

# Final Save
# !!! FIX 5: model.abstractor !!!
torch.save(model.abstractor.state_dict(), f"{config.save_path}/final_abstractor_epoch_{config.epochs}.pth")
model.llm.save_pretrained(f"{config.save_path}/final_lora_adapter")

wandb.finish()
print("Training Complete. Go write that paper.")

In [ ]:
for epoch in range(config.epochs):

  # set to training
  model.train()

  pbar = tqdm(train_loader)
  total_loss = 0

  # iterate train loader
  for step, batch in enumerate(pbar):
    pixel_values = batch["pixel_values"].to(device, dtype=torch.bfloat16)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    # Use mixed precision
    with torch.amp.autocast(device=device, dtype=torch.bfloat16):
      outputs = model(pixel_values, input_ids, attention_mask, labels)
      loss = outputs.loss
      loss = loss / config.grad_accumulation

    scaler.scale(loss).backward()

    # Use gradient accumulation
    if (step + 1) % config.grad_accumulation == 0:
      scaler.unscale_(optimizer)
      torch.nn.init.clip_grad_norm_(model.projection.parameters(), 1.0)

      scaler.update(optimizer)
      scaler.update()
      scheduler.update()
      optimizer.zero_grad()

    total_loss += loss.item()
    pbar.set_description(f"Epoch: {epoch + 1} | Loss: {loss.item() * config.grad_accumulation:.4f} | PPL: {math.exp(loss.item() * config.grad_accumulation):.4f}")

  # Logging on validation dataset
  if (epoch + 1) % config.log_step == 0:

    # set to evaluation state in validation part
    model.eval()

    val_total_loss = 0

    with torch.no_grad():

    # iterate val loader
    for batch in tqdm(val_loader, desc="Validation"):

      pixel_values = batch["pixel_values"].to(device, dtype=torch.bfloat16)
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      labels = batch = batch["labels"].to(device)

      # Use mixed precision
      with torch.amp.autocast(device=device, dtype=torch.bfloat16):
        outputs = model(pixel_values, input_ids, attention_mask, labels)
        val_loss = outputs.loss

      # compute validation loss
      val_total_loss += val_loss.item()

    avg_val_loss = val_total_loss / len(val_loader)
    avg_val_ppl = math.exp(avg_val_loss)

    print(f"Epoch: {epoch + 1} | Val_Loss: {avg_val_loss} | Val_PPL: {avg_val_ppl}")

  # Compute training loss
  avg_loss = total_loss / len(train_loader)
  avg_ppl = math.exp(avg_loss)
  print("-----" * 20)
  print(f"Epoch: {epoch + 1} | Loss: {avg_loss} | PPL: {avg_ppl}")

  # Save each epoch
  torch.save({
      "epoch": {epoch + 1},
      "model_projection_state_dict": model.projector.state_dict(),
      "model_decoder_state_dict": model.llm.state_dict(),
      "optimizer_state_dict": optimizer.state_dict(),
      "scaler_state_dict": scaler.state_dict(),
      "loss": avg_loss,
      "ppl": avg_ppl,
      "val_loss": avg_val_loss,
      "val_ppl": avg_val_ppl
  }, f"{confing.save_path}/projector_epoch_{epoch+1}.pt")

# Evaluate

In [ ]:
model.eval()

losses = 0

for batch in test_loader:
  with torch.no_grad():
    pixel_values, input_ids, attention_mask, labels = [b.to(device) for batch]

    with torch.amp.autocast(torch.bfloat16):
      outs = model(pixel_values, input_ids, attention_mask, labels)

    loss += outs.loss.item()

print(f"Loss: {loss}, ppl: {math.exp(loss)}")

# Inference

In [ ]:
def inference_check(image, prompt="Суретте не көріп тұрсың?"):
  try:
    image = Image.open(image).convert("RGB")
  except Exception as e:
    print(f"Error at loading image {image}: {e}")

  pixel_values = processor(images=[image], return_tensors="pt").squeeze(0)
  image_features = vision_model(**pixel_values)
  image_features = model.linear_projection(image_features)

  text_inputs = tokenizer(
      prompt,
      max_length=128,
      truncation=True,
      padding="max_length",
      return_tensors="pt"
  )

  input_ids = text_inputs.input_ids
  attention_mask = text_inputs.attention_mask

  labels = input_ids.clone()
  labels[labels == tokenizer.pad_token_id] = -100

  outs = model(pixel_values, text=input_ids, attention_mask=attention_mask, labels=labels)

  return tokenizer.decode(outs[0][outs["input_ids"].shape[-1]:])